In [37]:
# pip install chromadb ollama pypdf

In [1]:
import chromadb
import ollama
from pypdf import PdfReader

In [2]:
# make client and a colllection

demo_client = chromadb.Client()                      # lives in memory; gone when the kernel restarts
# demo_client.delete_collection("demo")
demo = demo_client.create_collection(
    name="demo",
    metadata={"hnsw:space": "cosine"}
)

print(demo.name, demo. count())

demo 0


In [5]:
# Add items
# We use ollama.embed to turn text into vectors

sentences = [
    "The Python course runs for 12 weeks.",
    "Students need 80 percent attendance to sit the final exam.",
    "The registration fee is paid at the front desk.",
    "The library is open from 7 am to 5 pm.",
]
vectors = ollama.embed(model="nomic-embed-text", input=sentences)["embeddings"]
 
demo.add(
    ids=["a", "b", "c", "d"],
    documents=sentences,
    embeddings=vectors,
    metadatas=[{"page": 1}, {"page": 1}, {"page": 2}, {"page": 3}],
)
print("Items stored:", demo.count())

Items stored: 4


In [10]:
question = "Tell me something about going daily?"
augmented = "How much attendance is required for the course?"
q_vector_1 = ollama.embed(model="nomic-embed-text", input=question)["embeddings"][0]
q_vector_2 = ollama.embed(model="nomic-embed-text", input=augmented)["embeddings"][0]

In [11]:
result = demo.query(query_embeddings=[q_vector_1], n_results=2)
print(result)

{'ids': [['d', 'c']], 'embeddings': None, 'documents': [['The library is open from 7 am to 5 pm.', 'The registration fee is paid at the front desk.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'page': 3}, {'page': 2}]], 'distances': [[0.5509727001190186, 0.6361174583435059]]}


In [13]:
print(demo.get(ids=["d"]))

{'ids': ['d'], 'embeddings': None, 'documents': ['The library is open from 7 am to 5 pm.'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'page': 3}]}


In [14]:
result = demo.get(ids=["d"], include=["embeddings", "documents"])
vector = result["embeddings"][0]
print("Text:  ", result["documents"][0])
print("Length:", len(vector), "numbers")
print("First 8:", vector[:8])

Text:   The library is open from 7 am to 5 pm.
Length: 768 numbers
First 8: [-0.05570194  0.0244479  -0.18042253 -0.05879634 -0.00802081  0.02729619
 -0.01009061 -0.03777778]


In [16]:
# Search with query

q_vector = ollama.embed(model="nomic-embed-text", input="How many weeks is the course?")["embeddings"][0]
result = demo.query(query_embeddings=[q_vector], n_results=2)
print(result)

{'ids': [['a', 'd']], 'embeddings': None, 'documents': [['The Python course runs for 12 weeks.', 'The library is open from 7 am to 5 pm.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'page': 1}, {'page': 3}]], 'distances': [[0.2843589782714844, 0.5137279629707336]]}


In [17]:
print(result["documents"][0])
print(result["metadatas"][0])
print(result["distances"][0])

['The Python course runs for 12 weeks.', 'The library is open from 7 am to 5 pm.']
[{'page': 1}, {'page': 3}]
[0.2843589782714844, 0.5137279629707336]


In [20]:
# Step 1: Load the PDF

reader = PdfReader("./sample_handbook.pdf")

pages = []
for page in reader.pages:
    text = page.extract_text().replace("\n", " ")
    pages.append(text)
print("Pages:", len(pages))
print(pages[0][:300])

Pages: 4
Himal Tech Institute Student Handbook About the Courses Every course at Himal Tech Institute runs for twelve weeks. Classes are held from Monday to Friday, from ten in the morning to one in the afternoon. Each class group has no more than twenty students so that trainers can give personal help. The 


In [21]:
# Step 2: Split into chunks

text = pages[0]
print(text[0:500]) # chunk 1: characters 0 to 500
print("-" * 40)
print(text[450:950]) # chunk 2: starts 50 early, so 50 characters overlap

Himal Tech Institute Student Handbook About the Courses Every course at Himal Tech Institute runs for twelve weeks. Classes are held from Monday to Friday, from ten in the morning to one in the afternoon. Each class group has no more than twenty students so that trainers can give personal help. The institute offers courses in web design, data analysis and office skills. New batches start at the beginning of every month. Trainers are industry professionals who also work on real projects. Class no
----------------------------------------
essionals who also work on real projects. Class notes are shared on the student portal after each session. Admissions Applicants must be at least sixteen years old. To apply, bring a citizenship card or school ID, two passport size photos and your latest mark sheet. A short interview is held before admission is confirmed. Admission decisions are shared within three working days. Students who have taken a course at the institute before can skip the interv

In [25]:
chunk_size = 500
overlap = 50
step = chunk_size - overlap # move forward 450 each time
chunks = []
page_numbers = []

for page_number, text in enumerate(pages, start=1):
    for start in range(0, len(text), step):
        chunks.append(text[start : start + chunk_size])
        page_numbers.append(page_number)
        
print("Number of chunks:", len(chunks))
print(chunks[0])

Number of chunks: 11
Himal Tech Institute Student Handbook About the Courses Every course at Himal Tech Institute runs for twelve weeks. Classes are held from Monday to Friday, from ten in the morning to one in the afternoon. Each class group has no more than twenty students so that trainers can give personal help. The institute offers courses in web design, data analysis and office skills. New batches start at the beginning of every month. Trainers are industry professionals who also work on real projects. Class no


In [26]:
print(chunks[1])

essionals who also work on real projects. Class notes are shared on the student portal after each session. Admissions Applicants must be at least sixteen years old. To apply, bring a citizenship card or school ID, two passport size photos and your latest mark sheet. A short interview is held before admission is confirmed. Admission decisions are shared within three working days. Students who have taken a course at the institute before can skip the interview. Applicants under eighteen need writte


In [27]:
# Step 3: Embed the chunks and store them

client = chromadb.Client()
try:
    client.delete_collection("handbook")
except:
    pass
collection = client.create_collection(
    name="handbook",
    metadata={"hnsw:space": "cosine"})

# 1) turn every chunk into a vector
vectors = ollama.embed(model="nomic-embed-text", input=chunks)["embeddings"]

# 2) give every chunk an id and a page label
ids = []
metadatas = []
for i in range(len(chunks)):
    ids.append("chunk_" + str(i))
    metadatas.append({"page": page_numbers[i]})

# 3) store everything
collection.add(ids=ids, documents=chunks, embeddings=vectors, metadatas=metadatas)
print("Chunks stored:", collection.count())

Chunks stored: 11


In [28]:
# Step 4: Retrieve the closest chunks

question = "How long does each course run?"
q_vector = ollama.embed(model="nomic-embed-text", input=question)["embeddings"][0]
found = collection.query(query_embeddings=[q_vector], n_results=3)

passages = found["documents"][0]
pages_found = found["metadatas"][0]

for i in range(len(passages)):
    print("Page", pages_found[i]["page"], "|", passages[i])
    print("-" * 40)

Page 1 | Himal Tech Institute Student Handbook About the Courses Every course at Himal Tech Institute runs for twelve weeks. Classes are held from Monday to Friday, from ten in the morning to one in the afternoon. Each class group has no more than twenty students so that trainers can give personal help. The institute offers courses in web design, data analysis and office skills. New batches start at the beginning of every month. Trainers are industry professionals who also work on real projects. Class no
----------------------------------------
Page 4 | Certificates Certificates are issued within two weeks after the results are published. Each certificate carries a unique number so that employers can verify it. A replacement for a lost certificate costs one thousand rupees. Results are published on the student portal and on the notice board. Certificates can be collected from the front desk or sent by post for a small courier charge. Labs and Facilities Laptops are provided in the comp

In [29]:
# Step 5: Build the prompt and ask the model

context = "\n\n".join(passages)
prompt = f"""Answer the question using only the passages below. If the answer is not in the passages, say: I do not know.
Passages:
{context}

Question: {question}"""
print(prompt)

Answer the question using only the passages below.
If the answer is not in the passages, say: I do not know.
Passages:
Himal Tech Institute Student Handbook About the Courses Every course at Himal Tech Institute runs for twelve weeks. Classes are held from Monday to Friday, from ten in the morning to one in the afternoon. Each class group has no more than twenty students so that trainers can give personal help. The institute offers courses in web design, data analysis and office skills. New batches start at the beginning of every month. Trainers are industry professionals who also work on real projects. Class no

Certificates Certificates are issued within two weeks after the results are published. Each certificate carries a unique number so that employers can verify it. A replacement for a lost certificate costs one thousand rupees. Results are published on the student portal and on the notice board. Certificates can be collected from the front desk or sent by post for a small courier

In [30]:
reply = ollama.chat(
model="llama3.2:3b",
messages=[{"role": "user", "content": prompt}],
options={"temperature": 0},
)
print(reply["message"]["content"])

Each course at Himal Tech Institute runs for twelve weeks.


In [ ]:
# Step 6: Wrap it up and ask your own questions

def ask(question):
    q_vector = ollama.embed(model="nomic-embed-text", input=question)["embeddings"][0]
    found = collection.query(query_embeddings=[q_vector], n_results=3)
    context = "\n\n".join(found["documents"][0])
    
    prompt = (
    "Answer the question using only the passages below. "
    "If the answer is not in the passages, say: I do not know.\n\n"
    "Passages:\n" + context + "\n\nQuestion: " + question
    )
    
    reply = ollama.chat(
    model="llama3.2:3b",
    messages=[{"role": "user", "content": prompt}],
    options={"temperature": 0},
    )
    
    return reply["message"]["content"]
print(ask("What attendance is needed to sit the final exam?"))